# Notebook for testing Database Operations

This notebook allows you to connect to a database using the `DatabaseSessionManager` and test database operations, select and queries.

To use this notebook you need to set the `DATABASE_URI` env variable to you database connection string: 

```export DATABASE_URI=postgresql://<user_name>:<password>@<host>:<local_port>/<database_name>```

> NOTE: Jupyter kernels inherit variables from parent server process at STARTUP.  If you created this enviornmental variable _after_ launching Jupyter, you will need to restart the kernel.  If this is too finicky for you (inside VSCode, for example).  Copy the `sample.env` file in this directory `development/genomicsdb-schema` to `.env` and then edit the `DATABASE_URI` value.

The `DATABASE_URI` should then be obtained using our `Settings` object, just so that whole notebook works within our project infrastrcuture.

In [9]:
# get DATABASE URI

from niagads.settings.core import CustomSettings

class Settings(CustomSettings):
    DATABASE_URI: str

In [10]:
# test a connection string 
from niagads.database.session import DatabaseSessionManager

manager = DatabaseSessionManager(connection_string=Settings.from_env().DATABASE_URI)
await manager.test_connection()

True

In [11]:
from niagads.database.genomicsdb.schema.reference.externaldb import ExternalDatabase
from sqlalchemy import or_, select

db_name = 'kegg'

async with manager.session_ctx() as session:
    stmt = select(ExternalDatabase.name, ExternalDatabase.version).where(
    or_(
        ExternalDatabase.name.ilike(f"%{db_name}%"),
        ExternalDatabase.database_key == db_name.upper(),
    )
)
    result = (await session.execute(stmt)).mappings().all()
    print(result)

[{'name': 'KEGG: Kyoto Encyclopedia of Genes and Genomes', 'version': '117.0'}]
